In [ ]:
from src.data import load_qrels_from_path
import ir_datasets
from src.data import DATA_DIR_PROCESSED
from topic_gen.evaluate import QrelsEvaluator, CohenKappa, MeanAverageError, AreaUnderReceiver,  binarize_qrels
import pandas as pd

from topic_gen import logger
# logger.setLevel("DEBUG")

In [2]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_PROCESSED / "qrels"

predictions, names, metadata = load_qrels_from_path(BASE_DIR)

[topic_gen] [WARNING] (data.py:51) Metadata not found for result 2025-11-19_19:28:50, skipping...


In [3]:
# binarize qrels
predictions = [binarize_qrels(qrels) for qrels in predictions]

In [4]:
# Evaluate qrels
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=binarize_qrels(ir_datasets.load(
        "disks45/nocr/trec-robust-2004").qrels_iter()),
    measures=[CohenKappa(), MeanAverageError(), AreaUnderReceiver()],
    bootstrap=20,
    names=names)

[topic_gen] [WARNING] (evaluate.py:284) Missing qrels! 12/2939  qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:284) Missing qrels! 8/2943  qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:284) Missing qrels! 11/2940  qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:284) Missing qrels! 4/2947  qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:284) Missing qrels! 10/2941  qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:284) Missing qrels! 24/2927  qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:284) Missing qrels! 2/2949  qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:284) Missing qrels! 15/2936  qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.py:284) Missing qrels! 9/2942  qrels in references but not in predictions.
[topic_gen] [WARNING] (evaluate.

In [16]:
qrels = ir_datasets.load("disks45/nocr/trec-robust-2004").qrels_iter()

In [17]:
from ir_measures.util import QrelsConverter
q = QrelsConverter(qrels).as_dict_of_dict()

In [ ]:
def format_score(row):
    return f"{row['value']:.2f} ± {row['ci']:.2f}"


df = pd.DataFrame(res)
df["score"] = df.apply(format_score, axis=1)  # format score with ci
df = df.pivot(index="name", columns="measure",
              values="score").reset_index()  # pivot table
df = df.merge(metadata, left_on="name", right_on="date")  # merge metadata

In [9]:
from statsmodels.stats import inter_rater as irr
import numpy as np
import krippendorff as kd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import List, Union
import pandas as pd
import ir_measures
import os


def agreement(dfs: List[pd.DataFrame]) -> Union[float, float]:
    df = dfs[0]
    rating_cols = ["relevance"]
    for idx, d in enumerate(dfs[1:]):
        df = df.merge(d, on=["query_id", "doc_id"],
                      suffixes=("", f"_r{idx+1}"))

        rating_cols.append(f"relevance_r{idx+1}")
    agg = irr.aggregate_raters(df[rating_cols])
    # fleiss kappa
    fleiss = irr.fleiss_kappa(agg[0], method='fleiss')

    arrT = np.array(df[rating_cols]).transpose()
    krippendorf = kd.alpha(arrT, level_of_measurement='nominal')

    return fleiss, krippendorf

## Clarity
RQ: How is the agreement between different generated qrels using the generated topics?

### Inter-Rater Reliability Graded+Error (Label: 0,1,2,999)
- Fleiss Kappa shows a moderate (>0.4) to just barely substantial (>0.6) agreement across all three LLM annotators and topic prompts.


In [ ]:
results = []
for i, df_group in df.groupby(by=["topics_model", "topics_prompt", "topics_nqueries", "topics_ndocsneg", "topics_ndocspos"]):
    row = df_group.iloc[0]
    name = row["topics_model"] + "_" + row["topics_prompt"] + \
        "-" + str(int(row["topics_nqueries"])) + \
        "-" + str(int(row["topics_ndocsneg"])) + \
        "-" + str(int(row["topics_ndocspos"]))
    
    qrels_dfs = []
    for run in df_group["name"]:
        qrel = pd.DataFrame(ir_measures.read_trec_qrels(os.path.join(BASE_DIR, run, "qrels.csv.gz")))
        qrels_dfs.append(qrel)

    if len(qrels_dfs) < 2:
        continue  # need at least two raters
    
    fleiss, krippendorf = agreement(qrels_dfs)

    results.append({
        "name": name,
        "krippendorf": krippendorf,
        "fleiss": fleiss
    })
table = pd.DataFrame(results)
table

KeyError: 'topics_model'

### Inter-Rater Reliability Binary+Error (Label: 0,1,999)

In [ ]:
results = []
for i, df_group in df.groupby(by=["topics_model", "topics_prompt", "topics_nqueries", "topics_ndocsneg", "topics_ndocspos"]):
    row = df_group.iloc[0]
    name = row["topics_model"] + "_" + row["topics_prompt"] + \
        "-" + str(int(row["topics_nqueries"])) + \
        "-" + str(int(row["topics_ndocsneg"])) + \
        "-" + str(int(row["topics_ndocspos"]))
    qrels_dfs = []
    for run in df_group["name"]:
        qrel = pd.DataFrame(binarize_qrels(ir_measures.read_trec_qrels(
            os.path.join(BASE_DIR, run, "qrels.csv.gz"))))
        qrels_dfs.append(qrel)

    fleiss, krippendorf = agreement(qrels_dfs)

    results.append({
        "name": name,
        "krippendorf": krippendorf,
        "fleiss": fleiss
    })
df = pd.DataFrame(results)
df

### Cross Matrix (binary)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes_flat = axes.flatten()

groups = table.groupby(by=["prompt", "topics_prompt",
                       "topics_nqueries", "topics_ndocsneg", "topics_ndocspos"])

for ax, (key, df) in zip(axes_flat, groups):
    df = df.copy()

    row = df.iloc[0]
    name = row["topics_prompt"] + \
        "-" + str(int(row["topics_nqueries"])) + \
        "-" + str(int(row["topics_ndocsneg"])) + \
        "-" + str(int(row["topics_ndocspos"]))

    df["CohenKappa"] = df["CohenKappa"].apply(
        lambda x: float(str(x).split(" ± ")[0]))
    df["model"] = df["model"].str.replace("no-think", "")
    df["topics_model"] = df["topics_model"].str.replace("no-think", "")
    df_pivot = df[["name", "CohenKappa", "model", "topics_model"]].pivot(
        index="model",
        columns="topics_model",
        values="CohenKappa"
    )

    sns.heatmap(df_pivot,
                annot=True,
                cmap='YlGnBu',
                ax=ax,
                # vmin=0.5,
                # vmax=0.8
                )
    ax.set_xlabel('Topic Model')
    ax.set_ylabel('Qrel Model')
    ax.set_title(name)

if len(groups) < len(axes_flat):
    for i in range(len(groups), len(axes_flat)):
        fig.delaxes(axes_flat[i])